In [1]:
import sys
import warnings
sys.path.append('..')

import pandas as pd
import numpy as np
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

from src.config import TRAIN_PATH, TEST_PATH, RANDOM_SEED
from src.preprocessing import HousePricesPreprocessor
from src.validation import cross_validate

warnings.filterwarnings('ignore')

In [2]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

In [3]:
y = np.log1p(train['SalePrice'])
test_ids = test['Id']
train = train.drop(['SalePrice', 'Id'], axis=1)
test = test.drop('Id', axis=1)

# Preprocessing (без нормализации)
preprocessor = HousePricesPreprocessor(scale=False)
X = preprocessor.fit_transform(train, np.expm1(y))
X_test = preprocessor.transform(test)

# Выбросы
outlier_idx = X[(X['GrLivArea'] > 4000) & (np.expm1(y) < 200000)].index
X = X.drop(outlier_idx)
y = y.drop(outlier_idx)

print(f"X: {X.shape}, y: {y.shape}")

X: (1458, 99), y: (1458,)


In [4]:
model = CatBoostRegressor(
    iterations=1000,
    learning_rate=0.05,
    depth=6,
    random_seed=RANDOM_SEED,
    verbose=0
)
scores = cross_validate(model, X, y)

RMSE по фолдам: [0.1083 0.1276 0.106  0.1243 0.1182]
Среднее: 0.1169 ± 0.0085


In [5]:
for depth in [4, 6, 8, 10]:
    for lr in [0.01, 0.05, 0.1]:
        print(f"\ndepth={depth}, lr={lr}:")
        model = CatBoostRegressor(
            iterations=1000,
            learning_rate=lr,
            depth=depth,
            random_seed=RANDOM_SEED,
            verbose=0
        )
        cross_validate(model, X, y)


depth=4, lr=0.01:
RMSE по фолдам: [0.1106 0.1325 0.1076 0.1264 0.1234]
Среднее: 0.1201 ± 0.0095

depth=4, lr=0.05:
RMSE по фолдам: [0.1102 0.1315 0.1036 0.1225 0.1183]
Среднее: 0.1172 ± 0.0097

depth=4, lr=0.1:
RMSE по фолдам: [0.1126 0.1303 0.1075 0.126  0.1142]
Среднее: 0.1181 ± 0.0086

depth=6, lr=0.01:
RMSE по фолдам: [0.1103 0.1293 0.1067 0.1239 0.1219]
Среднее: 0.1184 ± 0.0085

depth=6, lr=0.05:
RMSE по фолдам: [0.1083 0.1276 0.106  0.1243 0.1182]
Среднее: 0.1169 ± 0.0085

depth=6, lr=0.1:
RMSE по фолдам: [0.1115 0.1274 0.107  0.1231 0.1186]
Среднее: 0.1175 ± 0.0074

depth=8, lr=0.01:
RMSE по фолдам: [0.1123 0.1283 0.1107 0.1215 0.1308]
Среднее: 0.1207 ± 0.0082

depth=8, lr=0.05:
RMSE по фолдам: [0.1111 0.1279 0.1102 0.1236 0.1247]
Среднее: 0.1195 ± 0.0074

depth=8, lr=0.1:
RMSE по фолдам: [0.1148 0.1311 0.115  0.1243 0.1287]
Среднее: 0.1228 ± 0.0068

depth=10, lr=0.01:
RMSE по фолдам: [0.1177 0.131  0.1181 0.1254 0.1405]
Среднее: 0.1265 ± 0.0085

depth=10, lr=0.05:
RMSE по фолд

In [6]:
for n_estimators in [500, 1000]:
    for max_depth in [4, 6, 8]:
        for lr in [0.01, 0.05, 0.1]:
            print(f"\nn_est={n_estimators}, depth={max_depth}, lr={lr}:")
            model = LGBMRegressor(
                n_estimators=n_estimators,
                max_depth=max_depth,
                learning_rate=lr,
                random_seed=RANDOM_SEED,
                verbose=-1
            )
            cross_validate(model, X, y)


n_est=500, depth=4, lr=0.01:
RMSE по фолдам: [0.1164 0.138  0.1164 0.1362 0.1249]
Среднее: 0.1263 ± 0.0093

n_est=500, depth=4, lr=0.05:
RMSE по фолдам: [0.1113 0.1332 0.1151 0.1308 0.1221]
Среднее: 0.1225 ± 0.0085

n_est=500, depth=4, lr=0.1:
RMSE по фолдам: [0.1159 0.1338 0.1202 0.1341 0.1251]
Среднее: 0.1258 ± 0.0072

n_est=500, depth=6, lr=0.01:
RMSE по фолдам: [0.1171 0.1337 0.1173 0.1384 0.1246]
Среднее: 0.1262 ± 0.0086

n_est=500, depth=6, lr=0.05:
RMSE по фолдам: [0.119  0.1337 0.1189 0.1377 0.1234]
Среднее: 0.1265 ± 0.0078

n_est=500, depth=6, lr=0.1:
RMSE по фолдам: [0.1231 0.135  0.1219 0.1384 0.125 ]
Среднее: 0.1287 ± 0.0067

n_est=500, depth=8, lr=0.01:
RMSE по фолдам: [0.1166 0.1319 0.1181 0.1379 0.1254]
Среднее: 0.1260 ± 0.0081

n_est=500, depth=8, lr=0.05:
RMSE по фолдам: [0.119  0.134  0.1177 0.1369 0.1248]
Среднее: 0.1265 ± 0.0077

n_est=500, depth=8, lr=0.1:
RMSE по фолдам: [0.1209 0.1353 0.1196 0.1392 0.1272]
Среднее: 0.1284 ± 0.0077

n_est=1000, depth=4, lr=0.01:


In [7]:
for n_estimators in [500, 1000]:
    for max_depth in [3, 5, 7]:
        for lr in [0.01, 0.05, 0.1]:
            print(f"\nn_est={n_estimators}, depth={max_depth}, lr={lr}:")
            model = XGBRegressor(
                n_estimators=n_estimators,
                max_depth=max_depth,
                learning_rate=lr,
                random_seed=RANDOM_SEED,
                verbosity=0
            )
            cross_validate(model, X, y)
            


n_est=500, depth=3, lr=0.01:
RMSE по фолдам: [0.1214 0.143  0.1173 0.1358 0.1267]
Среднее: 0.1288 ± 0.0094

n_est=500, depth=3, lr=0.05:
RMSE по фолдам: [0.1154 0.1389 0.1087 0.1281 0.1148]
Среднее: 0.1212 ± 0.0109

n_est=500, depth=3, lr=0.1:
RMSE по фолдам: [0.12   0.1394 0.1155 0.1286 0.1142]
Среднее: 0.1235 ± 0.0094

n_est=500, depth=5, lr=0.01:
RMSE по фолдам: [0.1226 0.1395 0.1166 0.1389 0.1236]
Среднее: 0.1282 ± 0.0093

n_est=500, depth=5, lr=0.05:
RMSE по фолдам: [0.1209 0.1384 0.1154 0.1371 0.1186]
Среднее: 0.1261 ± 0.0097

n_est=500, depth=5, lr=0.1:
RMSE по фолдам: [0.1202 0.1416 0.1167 0.1332 0.1177]
Среднее: 0.1259 ± 0.0098

n_est=500, depth=7, lr=0.01:
RMSE по фолдам: [0.1279 0.1463 0.1233 0.1447 0.1299]
Среднее: 0.1344 ± 0.0093

n_est=500, depth=7, lr=0.05:
RMSE по фолдам: [0.1268 0.1478 0.121  0.1444 0.1265]
Среднее: 0.1333 ± 0.0107

n_est=500, depth=7, lr=0.1:
RMSE по фолдам: [0.1263 0.1468 0.1211 0.1442 0.1279]
Среднее: 0.1333 ± 0.0103

n_est=1000, depth=3, lr=0.01:


Бустинги: итоги
- CatBoost  (depth=6, lr=0.05):              CV=0.1169
- XGBoost   (n_est=500, depth=3, lr=0.05):   CV=0.1212
- LightGBM  (n_est=500, depth=4, lr=0.05):   CV=0.1225